# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamaKousalya/FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv("../data/flyrank.csv")
df.head()


FileNotFoundError: [Errno 2] No such file or directory: '../data/flyrank.csv'

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Staleness
df["stale_flag"] = df["days_since_refresh"] > 180

# CTR vs position
df["ctr_flag"] = df["ctr"] < df.groupby("position")["ctr"].transform("median") * 0.75

# Volume quick-win
df["volume_flag"] = (
    (df["volume"] >= df["volume"].quantile(.75)) &
    (df["position"].between(4, 20))
)

display(df.groupby("stale_flag").size())
display(df.groupby("ctr_flag").size())
display(df.groupby("volume_flag").size())


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["score"] = (
    df["stale_flag"].astype(int) * 3 +
    df["ctr_flag"].astype(int) * 3 +
    df["volume_flag"].astype(int) * 4
)

df["reason_code"] = np.select(
    [df["volume_flag"], df["ctr_flag"], df["stale_flag"]],
    ["HIGH_VOLUME_QUICKWIN", "LOW_CTR_FOR_POSITION", "STALE_REFRESH"],
    default="NONE"
)

df["action_label"] = np.where(
    df["score"] >= 6, "OPTIMIZE",
    np.where(df["score"] >= 3, "REVIEW", "MONITOR")
)

queue = df.sort_values("score", ascending=False)
display(queue.head(10))


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10).copy()

top10["what_would_make_it_wrong"] = (
    "Signal may be noisy, data incomplete, or search intent different."
)

display(top10[[
    "action_label",
    "reason_code",
    "score",
    "what_would_make_it_wrong"
]])

queue.to_csv(
    "../outputs/baseline_action_score.csv",
    index=False
)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.